# Custom-encoder ExpertXAS sweep: all FEFF and VASP datasets

Train three random-initialized ExpertXAS heads for each of the eight FEFF datasets and both VASP datasets. Every model uses repaired/exported 64D features from the same `scratch_temp_seed42` custom encoder. Runs are selected only by validation eta; test evaluation is disabled until model selection is final.


## 1. Imports and experiment configuration


In [1]:
from __future__ import annotations

from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from lightning.pytorch import seed_everything
from torch.utils.data import DataLoader, TensorDataset

from omnixas.data.ml_data import MLData, MLSplits
from omnixas.model.training import PlModule
from omnixas.model.xasblock import XASBlock
from omnixas.model.xasblock_regressor import XASBlockRegressor

torch.set_float32_matmul_precision("high")
REPO_ROOT = Path.cwd().resolve()
while not ((REPO_ROOT / "pyproject.toml").exists() and (REPO_ROOT / "omnixas").exists()):
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Run this notebook from inside the OmniXAS repo")
    REPO_ROOT = REPO_ROOT.parent

DEFAULT_ENCODER_RUN = REPO_ROOT.parent / "training071326" / "m3gnetAll8FEFF" / "scratch_temp_seed42"
ENCODER_RUN = Path(os.environ.get("OMNIXAS_ENCODER_RUN", DEFAULT_ENCODER_RUN)).expanduser().resolve()
FEATURES_DIR = ENCODER_RUN / "features"
EXPERT_ROOT = ENCODER_RUN / "heads" / "expertXAS"
RESULTS_DIR = ENCODER_RUN / "analysis" / "expertxas_all_custom"

DATASETS = [
    "Ti_FEFF", "V_FEFF", "Cr_FEFF", "Mn_FEFF", "Fe_FEFF",
    "Co_FEFF", "Ni_FEFF", "Cu_FEFF", "Ti_VASP", "Cu_VASP",
]
HPARAMS = {
    "Ti_FEFF": {"batch_size": 32, "hidden_dims": [600, 600, 450]},
    "V_FEFF":  {"batch_size": 32, "hidden_dims": [600, 550, 450]},
    "Cr_FEFF": {"batch_size": 32, "hidden_dims": [450, 350, 150]},
    "Mn_FEFF": {"batch_size": 64, "hidden_dims": [500, 400, 300]},
    "Fe_FEFF": {"batch_size": 64, "hidden_dims": [450, 400, 450]},
    "Co_FEFF": {"batch_size": 32, "hidden_dims": [600, 550, 450]},
    "Ni_FEFF": {"batch_size": 32, "hidden_dims": [600, 300]},
    "Cu_FEFF": {"batch_size": 32, "hidden_dims": [600, 600, 400]},
    "Ti_VASP": {"batch_size": 64, "hidden_dims": [500, 600, 400]},
    "Cu_VASP": {"batch_size": 64, "hidden_dims": [550, 600, 450]},
}
SEEDS = [42, 43, 44]
INPUT_DIM, OUTPUT_DIM = 64, 141
DROPOUT = 0.5
INITIAL_LR = 1e-3
MAX_EPOCHS = 1000
EARLY_STOPPING_PATIENCE = 25
RUN_TRAINING = True
RESUME_EXISTING = True
RUN_FINAL_TEST = False

if set(DATASETS) != set(HPARAMS):
    raise ValueError("Every dataset must have exactly one ExpertXAS architecture")
for required in [ENCODER_RUN, FEATURES_DIR]:
    if not required.exists():
        raise FileNotFoundError(f"Missing required path: {required}")
repair_manifests = list((ENCODER_RUN / "analysis").glob("invalid_vasp_wrong_absorber_*/repair_manifest.json"))
if not repair_manifests:
    raise FileNotFoundError(f"No repaired VASP manifest under {ENCODER_RUN / 'analysis'}")
REPAIR_MANIFEST = max(repair_manifests, key=lambda p: p.stat().st_mtime)
repair_info = json.loads(REPAIR_MANIFEST.read_text(encoding="utf-8"))
if set(repair_info.get("repaired_tasks", [])) != {"Ti_VASP", "Cu_VASP"}:
    raise ValueError(f"Unexpected VASP repair manifest: {REPAIR_MANIFEST}")
for dataset in DATASETS:
    for split_name in ["train", "val", "test"]:
        for suffix in ["X", "y"]:
            data_file = FEATURES_DIR / f"{dataset}_{split_name}_{suffix}.txt"
            if not data_file.exists():
                raise FileNotFoundError(f"Missing custom-encoder data: {data_file}")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("encoder run:", ENCODER_RUN)
print("VASP repair manifest:", REPAIR_MANIFEST)
print("ExpertXAS runs:", len(DATASETS) * len(SEEDS), f"({len(SEEDS)} per dataset)")
print("final test enabled:", RUN_FINAL_TEST)


encoder run: /mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42
VASP repair manifest: /mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/analysis/invalid_vasp_wrong_absorber_20260721_092524/repair_manifest.json
ExpertXAS runs: 30 (3 per dataset)
final test enabled: False


## 2. Load and validate custom-encoder feature splits


In [2]:
def load_splits(dataset: str) -> MLSplits:
    parts = {}
    for split_name in ["train", "val", "test"]:
        X = np.atleast_2d(np.loadtxt(FEATURES_DIR / f"{dataset}_{split_name}_X.txt", dtype=np.float32))
        y = np.atleast_2d(np.loadtxt(FEATURES_DIR / f"{dataset}_{split_name}_y.txt", dtype=np.float32))
        if len(X) != len(y) or X.shape[1] != INPUT_DIM or y.shape[1] != OUTPUT_DIM:
            raise ValueError(f"Invalid {dataset} {split_name}: X={X.shape}, y={y.shape}")
        if not np.isfinite(X).all() or not np.isfinite(y).all():
            raise ValueError(f"Non-finite values in {dataset} {split_name}")
        parts[split_name] = MLData(X=X, y=y)
    return MLSplits(**parts)

splits = {dataset: load_splits(dataset) for dataset in DATASETS}
display(pd.DataFrame([
    {
        "dataset": dataset,
        "train": len(split.train.X), "val": len(split.val.X), "test": len(split.test.X),
        "input_dim": split.train.X.shape[1], "output_dim": split.train.y.shape[1],
    }
    for dataset, split in splits.items()
]))


,dataset,train,val,test,input_dim,output_dim
0,Ti_FEFF,5140,641,641,64,141
1,V_FEFF,8653,1080,1080,64,141
2,Cr_FEFF,2457,305,305,64,141
3,Mn_FEFF,13644,1704,1704,64,141
4,Fe_FEFF,9657,1205,1205,64,141
5,Co_FEFF,8605,1074,1074,64,141
6,Ni_FEFF,3471,432,432,64,141
7,Cu_FEFF,3340,416,416,64,141
8,Ti_VASP,3030,377,377,64,141
9,Cu_VASP,2584,321,321,64,141


## 3. ExpertXAS training and metric helpers


In [3]:
def run_dir(dataset: str, seed: int) -> Path:
    return EXPERT_ROOT / dataset / "runs" / f"custom_encoder_seed{seed}"


def best_checkpoint(directory: Path) -> Path | None:
    checkpoints = sorted(directory.glob("best*.ckpt"))
    return checkpoints[0] if len(checkpoints) == 1 else None


def build_expert(directory: Path, dataset: str) -> XASBlockRegressor:
    hparams = HPARAMS[dataset]
    XASBlock.DROPOUT = DROPOUT
    return XASBlockRegressor(
        directory=str(directory), overwrite_save_dir=False,
        input_dim=INPUT_DIM, output_dim=OUTPUT_DIM,
        hidden_dims=hparams["hidden_dims"], batch_size=hparams["batch_size"],
        max_epochs=MAX_EPOCHS, early_stopping_patience=EARLY_STOPPING_PATIENCE,
        initial_lr=INITIAL_LR, min_lr=1e-4,
        use_lr_finder=True, use_early_stopping=True,
        monitor_metric="val_loss", shuffle=False, lr_scheduler="none",
    )


def predict(checkpoint: Path, dataset: str, X: np.ndarray) -> np.ndarray:
    XASBlock.DROPOUT = 0.0
    module = PlModule.load_from_checkpoint(
        checkpoint_path=str(checkpoint),
        model=XASBlock(INPUT_DIM, HPARAMS[dataset]["hidden_dims"], OUTPUT_DIM),
        lr=INITIAL_LR,
    ).to("cuda:0" if torch.cuda.is_available() else "cpu").eval()
    device = next(module.parameters()).device
    loader = DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32)), batch_size=1024)
    with torch.inference_mode():
        return np.concatenate([module(batch[0].to(device)).cpu().numpy() for batch in loader])


def eta_metrics(split: MLSplits, split_name: str, predictions: np.ndarray) -> dict[str, float]:
    targets = getattr(split, split_name).y
    mse = np.mean((targets - predictions) ** 2, axis=1)
    baseline = np.mean((targets - split.train.y.mean(axis=0, keepdims=True)) ** 2, axis=1)
    return {"eta": float(np.median(baseline) / np.median(mse)), "median_mse": float(np.median(mse))}


## 4. Train 30 random-initialized ExpertXAS heads

Each model is initialized from scratch. No UniversalXAS or tuned-head checkpoint is loaded.


In [4]:
if RUN_TRAINING:
    for dataset in DATASETS:
        for seed in SEEDS:
            directory = run_dir(dataset, seed)
            metadata = {
                "model_family": "ExpertXAS", "initialization": "random",
                "dataset": dataset, "seed": seed,
                "encoder_run": str(ENCODER_RUN), "feature_source": str(FEATURES_DIR),
                "repair_manifest": str(REPAIR_MANIFEST) if dataset.endswith("_VASP") else None,
                "hidden_dims": HPARAMS[dataset]["hidden_dims"],
                "batch_size": HPARAMS[dataset]["batch_size"],
                "dropout": DROPOUT, "initial_lr": INITIAL_LR,
                "max_epochs": MAX_EPOCHS, "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            }
            checkpoint = best_checkpoint(directory) if directory.exists() else None
            if checkpoint is not None and RESUME_EXISTING:
                saved = json.loads((directory / "settings.json").read_text(encoding="utf-8"))
                if saved != metadata:
                    raise ValueError(f"Existing run provenance mismatch: {directory}")
                print("SKIP", dataset, seed)
                continue
            if directory.exists():
                raise FileExistsError(f"Delete or quarantine incomplete run before retrying: {directory}")

            seed_everything(seed, workers=True)
            directory.mkdir(parents=True)
            (directory / "settings.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
            print("Training random-init ExpertXAS", dataset, "seed", seed)
            build_expert(directory, dataset).fit(splits[dataset])
            if best_checkpoint(directory) is None:
                raise RuntimeError(f"No unique best checkpoint: {directory}")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()


Seed set to 42


Training random-init ExpertXAS Ti_FEFF seed 42


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/Ti_FEFF/runs/custom_encoder_seed42 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`weights_only` was not set, defaulting to `False`.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to

Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=100` reached.
Restoring states from the checkpoint path at /mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/Ti_FEFF/runs/custom_encoder_seed42/.lr_find_93bf4780-0bee-43c9-99c4-8783e2893774.ckpt
Restored all states from the checkpoint at /mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/Ti_FEFF/runs/custom_encoder_seed42/.lr_find_93bf4780-0bee-43c9-99c4-8783e2893774.ckpt
Learning rate set to 0.00047863009232263843

  | Name  | Type     | Params | Mode  | FLOPs
---------------------------------------------------
0 | loss  | MSELoss  | 0      | train | 0    
1 | model | XASBlock | 736 K  | train | 0    
---------------------------------------------------
736 K     Trainable params
0         Non-trainable params
736 K     Total params
2.948     Total estimated model params size (MB)
16        Modules in train mode
0         Modules in eval mode
0         Total Flops
Rest

...

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

2026-07-21 14:55:04.286 | INFO     | omnixas.model.xasblock_regressor:fit:246 - Best models saved at /mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/Cu_VASP/runs/custom_encoder_seed44/best-model-epoch=423-val_loss=0.0051.ckpt
2026-07-21 14:55:04.287 | INFO     | omnixas.model.xasblock_regressor:fit:247 - Best validation loss: 0.005338781978935003


## 5. Select ExpertXAS runs using validation eta only

The three seeds are summarized for stability. The highest-validation seed is retained per dataset. Test data are not evaluated here.


In [5]:
validation_rows = []
for dataset in DATASETS:
    for seed in SEEDS:
        directory = run_dir(dataset, seed)
        checkpoint = best_checkpoint(directory) if directory.exists() else None
        if checkpoint is None:
            raise RuntimeError(f"Missing completed ExpertXAS run: {directory}")
        metrics_path = directory / "validation.json"
        if metrics_path.exists():
            metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        else:
            result = eta_metrics(splits[dataset], "val", predict(checkpoint, dataset, splits[dataset].val.X))
            metrics = {"val_eta": result["eta"], "val_median_mse": result["median_mse"]}
            metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
        validation_rows.append({
            "dataset": dataset, "seed": seed, "checkpoint": str(checkpoint), **metrics,
        })

validation_df = pd.DataFrame(validation_rows).sort_values(["dataset", "val_eta"], ascending=[True, False])
summary_df = (
    validation_df.groupby("dataset", as_index=False)["val_eta"]
    .agg(median_val_eta="median", mean_val_eta="mean", std_val_eta="std",
         min_val_eta="min", max_val_eta="max", runs="count")
    .sort_values("dataset")
)
selected_df = validation_df.groupby("dataset", sort=False).head(1).sort_values("dataset")
validation_df.to_csv(RESULTS_DIR / "candidate_validation_metrics.csv", index=False)
summary_df.to_csv(RESULTS_DIR / "validation_summary.csv", index=False)
selected_df.to_csv(RESULTS_DIR / "selected_by_validation.csv", index=False)
display(summary_df.style.format(precision=4))
display(selected_df.style.format(precision=4))


,dataset,median_val_eta,mean_val_eta,std_val_eta,min_val_eta,max_val_eta,runs
0,Co_FEFF,23.2437,23.1219,0.5733,22.4974,23.6245,3
1,Cr_FEFF,11.8505,11.8849,0.1393,11.7661,12.0383,3
2,Cu_FEFF,6.2275,6.1854,0.0818,6.0912,6.2376,3
3,Cu_VASP,6.7577,6.7791,0.0980,6.6936,6.8860,3
4,Fe_FEFF,12.4702,12.6401,0.3231,12.4373,13.0127,3
5,Mn_FEFF,23.7777,23.8252,0.1719,23.6821,24.0158,3
6,Ni_FEFF,13.5435,13.4673,0.1494,13.2952,13.5631,3
7,Ti_FEFF,9.4798,9.3961,0.1873,9.1816,9.5270,3
8,Ti_VASP,4.9645,5.0176,0.1183,4.9352,5.1531,3
9,V_FEFF,10.4463,10.4395,0.0578,10.3786,10.4937,3


,dataset,seed,checkpoint,val_eta,val_median_mse
17,Co_FEFF,44,/mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/Co_FEFF/runs/custom_encoder_seed44/best-model-epoch=255-val_loss=0.0024.ckpt,23.6245,0.0013
8,Cr_FEFF,44,/mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/Cr_FEFF/runs/custom_encoder_seed44/best-model-epoch=345-val_loss=0.0069.ckpt,12.0383,0.0047
23,Cu_FEFF,44,/mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/Cu_FEFF/runs/custom_encoder_seed44/best-model-epoch=237-val_loss=0.0027.ckpt,6.2376,0.0020
29,Cu_VASP,44,/mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/Cu_VASP/runs/custom_encoder_seed44/best-model-epoch=423-val_loss=0.0051.ckpt,6.8860,0.0030
13,Fe_FEFF,43,/mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/Fe_FEFF/runs/custom_encoder_seed43/best-model-epoch=251-val_loss=0.0056.ckpt,13.0127,0.0022
11,Mn_FEFF,44,/mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/Mn_FEFF/runs/custom_encoder_seed44/best-model-epoch=277-val_loss=0.0039.ckpt,24.0158,0.0020
19,Ni_FEFF,43,/mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/Ni_FEFF/runs/custom_encoder_seed43/best-model-epoch=187-val_loss=0.0029.ckpt,13.5631,0.0019
2,Ti_FEFF,44,/mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/Ti_FEFF/runs/custom_encoder_seed44/best-model-epoch=429-val_loss=0.0089.ckpt,9.5270,0.0054
24,Ti_VASP,42,/mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/Ti_VASP/runs/custom_encoder_seed42/best-model-epoch=735-val_loss=0.0481.ckpt,5.1531,0.0308
3,V_FEFF,42,/mnt/c/Users/anton/desktop/training071326/m3gnetAll8FEFF/scratch_temp_seed42/heads/expertXAS/V_FEFF/runs/custom_encoder_seed42/best-model-epoch=139-val_loss=0.0084.ckpt,10.4937,0.0052


## 6. Optional final test evaluation

Keep `RUN_FINAL_TEST = False` while changing the experiment. Enable it only after the validation-selected ExpertXAS protocol is final.


In [6]:
if not RUN_FINAL_TEST:
    print("Test evaluation skipped.")
else:
    artifact = RESULTS_DIR / "final_test_results.csv"
    if artifact.exists():
        raise FileExistsError(f"Final test results already exist: {artifact}")
    final_rows = []
    for selected in selected_df.to_dict(orient="records"):
        dataset = selected["dataset"]
        checkpoint = Path(selected["checkpoint"])
        result = eta_metrics(splits[dataset], "test", predict(checkpoint, dataset, splits[dataset].test.X))
        final_rows.append({
            "dataset": dataset, "seed": selected["seed"],
            "val_eta": selected["val_eta"], "test_eta": result["eta"],
            "test_median_mse": result["median_mse"], "checkpoint": str(checkpoint),
        })
    final_df = pd.DataFrame(final_rows).sort_values("dataset")
    final_df.to_csv(artifact, index=False)
    display(final_df.style.format(precision=4))


Test evaluation skipped.
